In [ ]:
!nvidia-smi

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs detected:", gpus)

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("GPU setup complete." if gpus else "No GPU detected.")

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Breast_Cancer_Project")
MODULE2_DIR = PROJECT_DIR / "module_2"

for folder in ["models", "results", "figures"]:
    (MODULE2_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Module 2 folder:", MODULE2_DIR)
print("Folders created successfully.")

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

print("Random seed:", SEED)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Setup complete.")

In [ ]:
from pathlib import Path
import subprocess
import tarfile

DATA_DIR = Path("/content/breakhis")
DATA_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_PATH = Path("/content/BreaKHis_v1.tar.gz")
DATA_URL = "https://www.inf.ufpr.br/vri/databases/BreaKHis_v1.tar.gz"

if not ARCHIVE_PATH.exists():
    subprocess.run(
        ["wget", "-c", DATA_URL, "-O", str(ARCHIVE_PATH)],
        check=True
    )

if not any(DATA_DIR.rglob("*.png")):
    with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
        archive.extractall(DATA_DIR, filter="data")

image_paths = sorted(DATA_DIR.rglob("*.png"))

print("Total images found:", len(image_paths))
print("Example filename:", image_paths[0].name if image_paths else "No images found")

In [ ]:
import pandas as pd

records = []

for path in image_paths:
    # Example: SOB_B_A-14-22549AB-100-001.png
    parts = path.stem.split("-")
    prefix = parts[0].split("_")

    if len(parts) != 5 or prefix[1] not in {"B", "M"}:
        raise ValueError(f"Unexpected filename: {path.name}")

    label = 0 if prefix[1] == "B" else 1

    records.append({
        "filepath": str(path),
        "filename": path.name,
        "label": label,
        "class_name": "benign" if label == 0 else "malignant",
        "patient_id": f"{parts[1]}-{parts[2]}",
        "magnification": int(parts[3]),
        "subtype": prefix[2]
    })

metadata = pd.DataFrame(records)

# Check that each patient has only one class label.
assert metadata.groupby("patient_id")["label"].nunique().max() == 1

metadata.to_csv(
    MODULE2_DIR / "results" / "image_metadata.csv",
    index=False
)

print("Total images:", len(metadata))
print("Unique patients:", metadata["patient_id"].nunique())

print("\nImages by class:")
print(metadata["class_name"].value_counts())

print("\nImages by magnification:")
print(metadata["magnification"].value_counts().sort_index())

display(metadata.head())

In [ ]:
check = metadata.copy()

# Preserve the class and subtype in a second identifier.
check["sample_group"] = check["filename"].apply(
    lambda name: "-".join(name.removesuffix(".png").split("-")[:3])
)

print("Patient IDs:", check["patient_id"].nunique())
print("Sample groups:", check["sample_group"].nunique())

group_counts = check.groupby("patient_id")["sample_group"].nunique()
shared_ids = group_counts[group_counts > 1].index

print("\nPatient IDs appearing in multiple sample groups:")
if len(shared_ids):
    display(
        check.loc[
            check["patient_id"].isin(shared_ids),
            ["patient_id", "sample_group", "class_name", "subtype"]
        ].drop_duplicates().sort_values("patient_id")
    )
else:
    print("None found.")

In [ ]:
from sklearn.model_selection import train_test_split

patients = metadata[["patient_id", "label"]].drop_duplicates()

train_patients, remaining_patients = train_test_split(
    patients,
    test_size=0.30,
    stratify=patients["label"],
    random_state=SEED
)

val_patients, test_patients = train_test_split(
    remaining_patients,
    test_size=0.50,
    stratify=remaining_patients["label"],
    random_state=SEED
)

train_df = metadata[
    metadata["patient_id"].isin(train_patients["patient_id"])
].copy()

val_df = metadata[
    metadata["patient_id"].isin(val_patients["patient_id"])
].copy()

test_df = metadata[
    metadata["patient_id"].isin(test_patients["patient_id"])
].copy()

# Confirm there is no patient overlap.
train_ids = set(train_df["patient_id"])
val_ids = set(val_df["patient_id"])
test_ids = set(test_df["patient_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
assert len(train_df) + len(val_df) + len(test_df) == len(metadata)

for name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    df.to_csv(
        MODULE2_DIR / "results" / f"{name}_metadata.csv",
        index=False
    )

    print(f"\n{name.upper()}")
    print("Patient groups:", df["patient_id"].nunique())
    print("Images:", len(df))
    print(df["class_name"].value_counts())

print("\nNo patient overlap. Split files saved.")

In [ ]:
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for row, class_name in enumerate(["benign", "malignant"]):
    samples = train_df[
        train_df["class_name"] == class_name
    ].sample(n=4, random_state=SEED)

    for col, (_, sample) in enumerate(samples.iterrows()):
        with Image.open(sample["filepath"]) as image:
            axes[row, col].imshow(image.convert("RGB"))

        axes[row, col].set_title(
            f"{class_name.capitalize()}\n"
            f"Patient: {sample['patient_id']} | "
            f"{sample['magnification']}×"
        )
        axes[row, col].axis("off")

plt.suptitle("BreaKHis: Sample Training Images", fontsize=16)
plt.tight_layout()

plt.savefig(
    MODULE2_DIR / "figures" / "training_image_samples.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.io.decode_png(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)

    # Keep pixel values in the 0–255 range for now.
    # Model-specific preprocessing will be added with the model.
    label = tf.cast(label, tf.float32)
    label = tf.reshape(label, (1,))

    return image, label

def make_dataset(df, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (
            df["filepath"].to_numpy(),
            df["label"].to_numpy(dtype=np.int32)
        )
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Pixel range:", float(tf.reduce_min(images)),
      "to", float(tf.reduce_max(images)))
print("Labels: 0 = benign, 1 = malignant")
print("Image pipelines ready.")

In [ ]:
from tensorflow.keras import layers, Model

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.10, seed=SEED + 1),
], name="augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = layers.Input(shape=(*IMAGE_SIZE, 3), name="image")
x = augmentation(inputs)
x = layers.Rescaling(1 / 127.5, offset=-1, name="preprocessing")(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30, seed=SEED)(x)
outputs = layers.Dense(
    1,
    activation="sigmoid",
    name="malignant_score"
)(x)

model = Model(inputs, outputs, name="breast_tissue_classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()
print("\nModel built successfully.")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].to_numpy()
)

class_weights = {
    int(label): float(weight)
    for label, weight in zip(classes, weights)
}

BEST_MODEL_PATH = MODULE2_DIR / "models" / "mobilenetv2_stage1_best.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_roc_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_roc_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        str(MODULE2_DIR / "results" / "stage1_training_log.csv")
    )
]

print("Class weights:", class_weights)

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

pd.DataFrame(history_stage1.history).to_csv(
    MODULE2_DIR / "results" / "stage1_history.csv",
    index=False
)

print("\nStage 1 training complete.")
print("Best model saved to:", BEST_MODEL_PATH)

In [ ]:
# Start from the best Stage 1 checkpoint.
model = tf.keras.models.load_model(str(BEST_MODEL_PATH))

base_model = model.get_layer("mobilenetv2_1.00_224")
base_model.trainable = True

# Allow only the last 30 layers to adapt.
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep batch-normalization layers frozen.
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile after changing which layers are trainable.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

print(
    "Trainable parameters:",
    sum(int(np.prod(w.shape)) for w in model.trainable_weights)
)
print("Fine-tuning learning rate: 0.00001")
print("Fine-tuning setup complete.")

In [ ]:
STAGE2_MODEL_PATH = (
    MODULE2_DIR / "models" / "mobilenetv2_stage2_best.keras"
)

callbacks_stage2 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(STAGE2_MODEL_PATH),
        monitor="val_roc_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_roc_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_roc_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        str(MODULE2_DIR / "results" / "stage2_training_log.csv")
    )
]

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks_stage2
)

pd.DataFrame(history_stage2.history).to_csv(
    MODULE2_DIR / "results" / "stage2_history.csv",
    index=False
)

print("\nFine-tuning complete.")
print("Best Stage 2 model saved to:", STAGE2_MODEL_PATH)
print(
    "Best Stage 2 validation ROC-AUC:",
    round(max(history_stage2.history["val_roc_auc"]), 4)
)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

final_model = tf.keras.models.load_model(
    str(STAGE2_MODEL_PATH)
)

val_scores = final_model.predict(val_ds, verbose=1).ravel()
val_labels = val_df["label"].to_numpy()

assert len(val_scores) == len(val_labels)
assert np.isfinite(val_scores).all()

validation_predictions = val_df.copy()
validation_predictions["malignant_score"] = val_scores

validation_predictions.to_csv(
    MODULE2_DIR / "results" / "validation_predictions.csv",
    index=False
)

print("\nValidation images:", len(val_scores))
print("ROC-AUC:", round(roc_auc_score(val_labels, val_scores), 4))
print(
    "Average precision:",
    round(average_precision_score(val_labels, val_scores), 4)
)
print("Validation predictions saved.")

In [ ]:
from sklearn.metrics import (
    roc_curve,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import json

fpr, tpr, thresholds = roc_curve(val_labels, val_scores)

# Maximizing TPR - FPR also maximizes balanced accuracy.
valid = np.isfinite(thresholds) & (thresholds >= 0) & (thresholds <= 1)
candidate_indices = np.flatnonzero(valid)
best_index = candidate_indices[np.argmax((tpr - fpr)[valid])]

SELECTED_THRESHOLD = float(thresholds[best_index])

rows = []

for name, threshold in [
    ("Default", 0.5),
    ("Validation-selected", SELECTED_THRESHOLD)
]:
    predictions = (val_scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        val_labels, predictions, labels=[0, 1]
    ).ravel()

    rows.append({
        "threshold_type": name,
        "threshold": threshold,
        "accuracy": accuracy_score(val_labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            val_labels, predictions
        ),
        "malignant_recall": recall_score(
            val_labels, predictions, zero_division=0
        ),
        "benign_recall": tn / (tn + fp),
        "precision": precision_score(
            val_labels, predictions, zero_division=0
        ),
        "f1": f1_score(val_labels, predictions, zero_division=0)
    })

threshold_comparison = pd.DataFrame(rows)

threshold_comparison.to_csv(
    MODULE2_DIR / "results" / "validation_threshold_comparison.csv",
    index=False
)

with open(MODULE2_DIR / "models" / "threshold_config.json", "w") as f:
    json.dump({
        "threshold": SELECTED_THRESHOLD,
        "selection_dataset": "validation",
        "selection_metric": "image_level_balanced_accuracy",
        "positive_class": "malignant",
        "positive_label": 1,
        "decision_rule": "score >= threshold"
    }, f, indent=2)

display(threshold_comparison.round(4))
print("Selected threshold:", round(SELECTED_THRESHOLD, 6))
print("Threshold configuration saved.")

In [ ]:
test_scores = final_model.predict(test_ds, verbose=1).ravel()
test_labels = test_df["label"].to_numpy()

assert len(test_scores) == len(test_labels)
assert np.isfinite(test_scores).all()

test_predictions = (
    test_scores >= SELECTED_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    test_labels, test_predictions, labels=[0, 1]
).ravel()

test_metrics = {
    "threshold": SELECTED_THRESHOLD,
    "images": len(test_labels),
    "patient_groups": int(test_df["patient_id"].nunique()),
    "roc_auc": roc_auc_score(test_labels, test_scores),
    "average_precision": average_precision_score(
        test_labels, test_scores
    ),
    "accuracy": accuracy_score(test_labels, test_predictions),
    "balanced_accuracy": balanced_accuracy_score(
        test_labels, test_predictions
    ),
    "malignant_recall": recall_score(
        test_labels, test_predictions, zero_division=0
    ),
    "benign_recall": float(tn / (tn + fp)),
    "precision": precision_score(
        test_labels, test_predictions, zero_division=0
    ),
    "f1": f1_score(
        test_labels, test_predictions, zero_division=0
    ),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp)
}

test_results = test_df.copy()
test_results["malignant_score"] = test_scores
test_results["predicted_label"] = test_predictions

test_results.to_csv(
    MODULE2_DIR / "results" / "test_predictions.csv",
    index=False
)

with open(MODULE2_DIR / "results" / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

display(
    pd.DataFrame([test_metrics]).T.rename(columns={0: "value"})
)

print("Test evaluation saved.")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    test_labels,
    test_predictions,
    labels=[0, 1],
    display_labels=["Benign", "Malignant"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0]
)
axes[0].set_title(
    f"Test Confusion Matrix\nThreshold = {SELECTED_THRESHOLD:.4f}"
)

RocCurveDisplay.from_predictions(
    test_labels,
    test_scores,
    name="MobileNetV2",
    ax=axes[1]
)
axes[1].plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="gray",
    label="Chance"
)
axes[1].set_title("Test ROC Curve")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig(
    MODULE2_DIR / "figures" / "test_confusion_matrix_and_roc.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

print("Test evaluation figure saved.")

In [ ]:
patient_rows = []

for patient_id, group in test_results.groupby("patient_id"):
    true_label = int(group["label"].iloc[0])
    predictions = group["predicted_label"].to_numpy()

    patient_rows.append({
        "patient_id": patient_id,
        "true_class": "malignant" if true_label == 1 else "benign",
        "image_count": len(group),
        "mean_malignant_score": group["malignant_score"].mean(),
        "fraction_flagged_malignant": predictions.mean(),
        "image_accuracy": (
            predictions == group["label"].to_numpy()
        ).mean()
    })

patient_results = pd.DataFrame(patient_rows).sort_values(
    ["true_class", "patient_id"]
)

patient_results.to_csv(
    MODULE2_DIR / "results" / "test_results_by_patient.csv",
    index=False
)

display(patient_results.round(4))

print(
    "\nMean image accuracy across patient groups:",
    round(patient_results["image_accuracy"].mean(), 4)
)
print("Patient-group results saved.")

In [ ]:
stage1 = pd.DataFrame(history_stage1.history)
stage2 = pd.DataFrame(history_stage2.history)

combined = pd.concat([stage1, stage2], ignore_index=True)
epochs = np.arange(1, len(combined) + 1)
boundary = len(stage1) + 0.5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, title in [
    (axes[0], "loss", "Training and Validation Loss"),
    (axes[1], "roc_auc", "Training and Validation ROC-AUC")
]:
    ax.plot(epochs, combined[metric], label="Training")
    ax.plot(epochs, combined[f"val_{metric}"], label="Validation")
    ax.axvline(
        boundary,
        color="gray",
        linestyle="--",
        label="Fine-tuning begins"
    )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss" if metric == "loss" else "ROC-AUC")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(
    MODULE2_DIR / "figures" / "training_history.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

print("Training history figure saved.")

In [ ]:
FINAL_MODEL_PATH = (
    MODULE2_DIR / "models" / "breast_tissue_classifier.keras"
)

final_model.save(str(FINAL_MODEL_PATH))

model_config = {
    "model_file": FINAL_MODEL_PATH.name,
    "architecture": "MobileNetV2",
    "selected_checkpoint": STAGE2_MODEL_PATH.name,
    "image_size": list(IMAGE_SIZE),
    "color_mode": "RGB",
    "input_pixel_range": [0, 255],
    "preprocessing": "Rescaling to [-1, 1] included in model",
    "resize_method": "bilinear",
    "labels": {"0": "benign", "1": "malignant"},
    "threshold": SELECTED_THRESHOLD,
    "decision_rule": "score >= threshold",
    "threshold_selection": "validation image-level balanced accuracy",
    "dataset": "BreaKHis",
    "magnifications": [40, 100, 200, 400],
    "random_seed": SEED,
    "intended_use": "Educational research demonstration",
    "score_note": "Model score is not a calibrated cancer probability",
    "test_metrics": test_metrics
}

CONFIG_PATH = MODULE2_DIR / "models" / "model_config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(model_config, f, indent=2)

# Verify that the saved model reproduces the original predictions.
reloaded_model = tf.keras.models.load_model(str(FINAL_MODEL_PATH))

sample_images, _ = next(iter(val_ds))

original_scores = final_model(
    sample_images, training=False
).numpy()

reloaded_scores = reloaded_model(
    sample_images, training=False
).numpy()

np.testing.assert_allclose(
    original_scores,
    reloaded_scores,
    rtol=1e-5,
    atol=1e-6
)

print("Final model saved:", FINAL_MODEL_PATH)
print("Configuration saved:", CONFIG_PATH)
print("Reloaded model predictions match.")

In [ ]:
import shutil
from google.colab import files

ZIP_PATH = shutil.make_archive(
    "/content/module_2_export",
    "zip",
    root_dir=str(PROJECT_DIR),
    base_dir="module_2"
)

files.download(ZIP_PATH)

In [ ]:
from google.colab import files

sample = test_results.iloc[0]

print("Filename:", sample["filename"])
print("Dataset label:", sample["class_name"])
print("Colab model score:", round(float(sample["malignant_score"]), 6))
print(
    "Colab prediction:",
    "malignant" if sample["predicted_label"] == 1 else "benign"
)

files.download(sample["filepath"])

